In [58]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg, udf
from pyspark.sql.types import StringType

# Initialize
spark = SparkSession.builder.appName("UDF Example").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", 25),
    (2, "Bob", 12),
    (3, "Charlie", 15),
    (4, "David", 28),
    (5, "Eve", 32)
]
columns = ["Id", "Name", "Age"]
df = spark.createDataFrame(data, columns);

In [59]:
df.show()

+---+-------+---+
| Id|   Name|Age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 12|
|  3|Charlie| 15|
|  4|  David| 28|
|  5|    Eve| 32|
+---+-------+---+



In [57]:
def categorize_age(age):
    if age >= 18:
        return "adult"
    return "minor"

In [36]:
age_category_udf = udf(categorize_age, StringType())

In [37]:
df_method1 = df.withColumn("category", age_category_udf(col("age")))
print("method 1: Standard UDF via DataFrame API")
df_method1.show()

method 1: Standard UDF via DataFrame API
+---+-------+-----------+------+---+--------+
| Id|   Name| Department|Salary|Age|category|
+---+-------+-----------+------+---+--------+
|  1|  Alice|Engineering| 75000| 25|   adult|
|  2|    Bob|  Marketing| 60000| 12|   minor|
|  3|Charlie|Engineering| 80000| 15|   minor|
|  4|  David|      Sales| 65000| 28|   adult|
|  5|    Eve|  Marketing| 70000| 32|   adult|
+---+-------+-----------+------+---+--------+



In [61]:
def can_drive(age):
    return "can drive" if age >= 18 else "cannot drive"

can_drive_udf = udf(can_drive, StringType())

In [62]:
df_method2 = df.withColumn("driving_status", can_drive_udf(col("Age")))
print("Method 2: Driving eligibility UDF")
df_method2.show()

Method 2: Driving eligibility UDF
+---+-------+---+--------------+
| Id|   Name|Age|driving_status|
+---+-------+---+--------------+
|  1|  Alice| 25|     can drive|
|  2|    Bob| 12|  cannot drive|
|  3|Charlie| 15|  cannot drive|
|  4|  David| 28|     can drive|
|  5|    Eve| 32|     can drive|
+---+-------+---+--------------+



In [47]:
spark.udf.register("sql_category_age", categorize_age, StringType())
df.createOrReplaceTempView("people")

26/06/13 06:57:23 WARN SimpleFunctionRegistry: The function sql_category_age replaced a previously registered function.


In [65]:
sql_df = spark.sql("""
    SELECT
        Name,
        Age,
        sql_category_age(Age) AS Category
    FROM people
""")
sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   adult|
|    Bob| 12|   minor|
|Charlie| 15|   minor|
|  David| 28|   adult|
|    Eve| 32|   adult|
+-------+---+--------+

